# Pokemon PSA-like Grader & Market Estimator

Este proyecto consiste en desarrollar una aplicacion web y una API que permitan estimar el estado visual de una carta Pokemon a partir de una imagen subida por el usuario, consultar informacion real de mercado mediante la Pokemon TCG API y devolver una estimacion de valor ajustada segun la condicion detectada.

El objetivo no es certificar una nota PSA real, ya que PSA es una empresa de grading profesional que evalua fisicamente las cartas bajo criterios muy estrictos. El objetivo del proyecto es construir una estimacion visual tipo PSA, es decir, una aproximacion automatizada basada en imagen que clasifique la condicion de la carta y la traduzca a un rango orientativo de puntuacion.


## 1. Contexto del problema

En el mercado de cartas coleccionables, el valor de una carta no depende solo de que carta sea. Tambien influyen factores como:

- Rareza y set de la carta.
- Demanda actual del mercado.
- Estado fisico: bordes, esquinas, superficie, dobleces, manchas o desgaste.
- Si la carta esta graduada profesionalmente o no.

Para un usuario no experto, estimar el estado de una carta puede ser complicado. Este proyecto busca crear una herramienta sencilla que ayude a obtener una primera valoracion visual y economica.

La aplicacion seguira este flujo principal:

1. El usuario busca o introduce una carta Pokemon.
2. La aplicacion consulta datos reales en la Pokemon TCG API.
3. El usuario sube una imagen de su carta.
4. Un modelo de Computer Vision estima la condicion visual.
5. La API devuelve una puntuacion PSA-like y un precio ajustado.

## 2. Objetivos del proyecto

### Objetivo principal

Productivizar un modelo de Machine Learning o Deep Learning dentro de una API accesible publicamente, integrando datos externos y una interfaz visual sencilla.

### Objetivos especificos

- Construir una API REST con FastAPI.
- Crear endpoints de estado, busqueda de cartas, prediccion visual y tasacion.
- Conectar con la Pokemon TCG API para obtener datos reales de cartas y precios.
- Realizar un mini-EDA sobre cartas, sets, rarezas y precios disponibles.
- Entrenar un primer modelo baseline para clasificar la condicion visual.
- Traducir la condicion estimada a una puntuacion orientativa tipo PSA 1-10.
- Ajustar el precio base de mercado usando factores de condicion.
- Crear una interfaz Streamlit para subir imagenes y mostrar resultados.
- Desplegar el backend y dejar documentadas instrucciones de uso.

## 3. Alcance del MVP

Para mantener el proyecto realista, el MVP no intentara identificar automaticamente la carta desde la imagen. En la primera version, el usuario buscara la carta por nombre o ID, y la imagen se usara para estimar su condicion.

### Incluido en el MVP

- Busqueda de carta por ID o nombre.
- Consulta de precio base mediante Pokemon TCG API.
- Subida de imagen de la carta.
- Clasificacion de condicion visual.
- Estimacion PSA-like de 1 a 10.
- Calculo de precio ajustado.
- API documentada con Swagger.
- Interfaz basica con Streamlit.

### Fuera del MVP inicial

- Identificacion automatica de la carta desde la imagen.
- Certificacion PSA real.
- Deteccion precisa de defectos individuales como microaranazos o problemas de centrado.
- Sistema de usuarios o historico de valoraciones.

Estas funcionalidades quedan como posibles mejoras futuras.

## 4. Etiquetas del modelo

Aunque el resultado final se mostrara como una puntuacion PSA-like de 1 a 10, el primer modelo se planteara como un problema de clasificacion. Esto es mas realista que entrenar directamente una regresion exacta a nota PSA, porque es dificil conseguir imagenes etiquetadas con puntuaciones oficiales fiables.

Las clases iniciales seran:

| Label | Descripcion | Rango PSA-like orientativo |
|---|---|---|
| `damaged` | Carta con dobleces, roturas, manchas fuertes o mucho desgaste | 1-3 |
| `played` | Carta claramente usada, con bordes o esquinas desgastadas | 4-5 |
| `excellent` | Carta en buen estado, con pequenas marcas visibles | 6-7 |
| `near_mint` | Carta muy cuidada, con defectos pequenos | 8-9 |
| `mint` | Carta aparentemente perfecta en la imagen | 10 |

Esta estrategia permite construir un baseline funcional y despues mejorar el modelo con mas datos o tecnicas de vision por computador.

## 5. Fuentes de datos

### Pokemon TCG API

La Pokemon TCG API se usara para obtener informacion estructurada de cartas:

- ID oficial.
- Nombre.
- Set.
- Rareza.
- Imagen oficial.
- Precios de mercado cuando esten disponibles.

Esta API es adecuada para el EDA y para la tasacion base, pero no sirve por si sola para entrenar el modelo de dano, porque sus imagenes son imagenes oficiales limpias de catalogo.

### Dataset de imagenes reales

Para entrenar el modelo de condicion se necesitan fotos reales de cartas en diferentes estados. Posibles fuentes:

- Dataset propio creado manualmente.
- Imagenes publicas con etiquetas de condicion.
- Datasets de Kaggle relacionados con cartas Pokemon, si incluyen imagenes reales y no solo imagenes oficiales.
- Datos de marketplaces solo si su uso cumple terminos legales y condiciones de uso.

La calidad del dataset sera uno de los riesgos principales del proyecto. Un modelo sencillo con datos bien etiquetados puede ser mas util que un modelo complejo entrenado con datos ruidosos.

## 6. Mini-EDA previsto

El EDA se centrara en entender los datos disponibles desde la Pokemon TCG API:

- Numero de cartas disponibles por set.
- Distribucion de rarezas.
- Cartas con precio disponible frente a cartas sin precio.
- Distribucion de precios medios.
- Comparacion de precios por rareza o set.
- Ejemplos de cartas populares y sus precios.

Este analisis no entrena el modelo visual, pero aporta contexto de negocio y permite justificar como se obtiene el precio base antes de ajustar por condicion.

## 7. Estrategia de modelado

La estrategia sera incremental.

### Baseline inicial

El primer modelo priorizara simplicidad y explicabilidad:

- Redimensionado de imagenes.
- Normalizacion de pixeles.
- Clasificacion en las cinco etiquetas definidas.
- Evaluacion con accuracy, matriz de confusion y ejemplos visuales.

Como primera opcion se usara Transfer Learning con MobileNetV2 o EfficientNetB0, porque son arquitecturas ligeras y adecuadas para despliegues con recursos limitados.

### Mejoras posteriores

- Aumentar dataset.
- Data augmentation.
- Separar analisis de frontal y reverso de la carta.
- Detectar bordes y esquinas.
- Incorporar un score numerico continuo.
- Identificar automaticamente la carta desde la imagen.

## 8. Estimacion de precio ajustado

El precio base se obtendra desde la Pokemon TCG API cuando este disponible. Despues se aplicara un factor segun la condicion estimada.

| Label | Factor inicial propuesto |
|---|---:|
| `mint` | 1.00-8.00 |
| `played` | 0.70-0.40 |
| `damaged` | 0.30-0.10 |

Estos factores son una heuristica inicial, no una verdad universal. En una version avanzada se podrian calibrar con historicos reales de ventas por condicion.

## 9. Diseno de la API

La API se desarrollara con FastAPI.

Endpoints previstos:

- `GET /health`: comprueba que el servicio esta activo.
- `GET /api/v1/card/{card_id}`: consulta una carta por ID oficial.
- `GET /api/v1/search?name=charizard`: busca cartas por nombre.
- `POST /api/v1/grade`: recibe una imagen y devuelve condicion estimada.
- `POST /api/v1/appraise`: recibe carta e imagen, estima condicion y devuelve precio ajustado.

FastAPI generara documentacion interactiva en `/docs`, lo cual facilita la presentacion y las pruebas.

## 10. Arquitectura prevista

Componentes principales:

- Backend: FastAPI.
- Modelo: TensorFlow/Keras o alternativa ligera compatible con despliegue.
- API externa: Pokemon TCG API.
- Frontend: Streamlit.
- Despliegue: AWS EC2, Render o plataforma equivalente.

Flujo simplificado:

1. Streamlit envia una peticion al backend.
2. FastAPI consulta datos de la carta.
3. FastAPI procesa la imagen con el modelo.
4. FastAPI calcula la puntuacion PSA-like y el precio ajustado.
5. Streamlit muestra el resultado al usuario.

## 11. Riesgos y limitaciones

- La estimacion no equivale a una nota PSA oficial.
- La calidad del modelo dependera mucho del dataset de imagenes reales.
- Una sola foto puede no mostrar todos los defectos de la carta.
- La iluminacion, el enfoque y el angulo pueden afectar la prediccion.
- Los precios de mercado pueden no estar disponibles para todas las cartas.
- Los factores de ajuste por condicion seran inicialmente heuristicas.

Estas limitaciones se explicaran claramente en la documentacion para evitar prometer mas precision de la que el sistema puede ofrecer.

## 12. Plan de trabajo

### Fase 1: Base del proyecto

- Organizar estructura del repositorio.
- Completar `requirements.txt`.
- Crear endpoints iniciales de FastAPI.
- Integrar consulta basica con Pokemon TCG API.

### Fase 2: EDA

- Extraer muestra de cartas desde la API.
- Analizar sets, rarezas y precios.
- Guardar conclusiones visuales para la presentacion.

### Fase 3: Modelo baseline

- Reunir dataset inicial de imagenes reales.
- Preparar carpetas por clase.
- Entrenar modelo ligero.
- Evaluar resultados.
- Exportar modelo.

### Fase 4: Productivizacion

- Cargar modelo al iniciar la API.
- Crear endpoint de prediccion.
- Crear endpoint de tasacion.
- Crear frontend Streamlit.

### Fase 5: Despliegue y documentacion

- Desplegar backend.
- Documentar instrucciones locales.
- Anadir ejemplos de peticiones y respuestas.
- Preparar demo para presentacion.

## 13. Ejecucion local prevista

Para levantar el servidor en local:

```bash
uvicorn app.main:app --reload
```

Una vez levantado, la documentacion interactiva estara disponible en:

```text
http://127.0.0.1:8000/docs
```